# Autonomous Path Finder - Dataset EDA

This notebook performs a full exploratory data analysis of a YOLO-format hospital dataset.

Run cells top-to-bottom. The workflow is organized into:
- setup and dataset loading
- annotation/image feature engineering
- visual diagnostics
- automated findings summary


In [1]:
# ============================================
# 0) Setup
# ============================================

import os, glob, random
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

sns.set_theme(context='talk', style='whitegrid')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('Ready.')


Ready.


In [ ]:
# ============================================
# 1) Configure dataset root
# ============================================
# If set to None, the notebook tries to auto-detect from /kaggle/input.
MANUAL_DATA_ROOT = Path('/home/ltgwgeorge/Desktop/Business/ML/ML-auto-wheels/Hospital.v1-hospitaldata.yolov8')

def autodetect_data_root() -> Path | None:
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists():
        return None
    matches = list(kaggle_input.rglob('data.yaml'))
    return matches[0].parent if matches else None

DATA_ROOT = MANUAL_DATA_ROOT if MANUAL_DATA_ROOT is not None else autodetect_data_root()
YAML_PATH = DATA_ROOT / 'data.yaml' if DATA_ROOT is not None else None

assert YAML_PATH is not None and YAML_PATH.exists(), (
    'Could not locate data.yaml. Set MANUAL_DATA_ROOT to your dataset folder.'
)

print('DATA_ROOT:', DATA_ROOT)
print('YAML_PATH:', YAML_PATH)


In [ ]:
# ============================================
# 2) Read YAML metadata
# ============================================
with open(YAML_PATH, 'r') as f:
    data_cfg = yaml.safe_load(f)

train_rel = data_cfg.get('train')
val_rel = data_cfg.get('val', data_cfg.get('valid'))
test_rel = data_cfg.get('test')

assert train_rel is not None, 'data.yaml is missing `train` path.'
assert val_rel is not None, 'data.yaml is missing `val`/`valid` path.'
assert test_rel is not None, 'data.yaml is missing `test` path.'

names = data_cfg.get('names', [])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
nc = int(data_cfg.get('nc', len(names)))

print('nc:', nc)
print('names:', names)
print('train:', train_rel)
print('val:', val_rel)
print('test:', test_rel)


In [ ]:
# ============================================
# 3) Resolve split paths + build dataframes
# ============================================

def resolve_images_dir(split_rel_path: str) -> Path:
    return (YAML_PATH.parent / split_rel_path).resolve()

def find_images_dir(root: Path, split_name: str):
    candidates = []
    for p in root.rglob('*'):
        if p.is_dir():
            parts = [x.lower() for x in p.parts]
            if split_name.lower() in parts and 'images' in parts:
                candidates.append(p)
    return candidates[0] if candidates else None

def pick_images_dir(split_rel, split_name, fallback_name=None):
    candidate = resolve_images_dir(split_rel)
    if candidate.exists():
        return candidate
    found = find_images_dir(DATA_ROOT, split_name)
    if found is not None:
        return found
    if fallback_name is not None:
        return find_images_dir(DATA_ROOT, fallback_name)
    return None

train_images_dir = pick_images_dir(train_rel, 'train')
val_images_dir = pick_images_dir(val_rel, 'valid', fallback_name='val')
test_images_dir = pick_images_dir(test_rel, 'test')

print('Detected paths:')
print(' train_images_dir:', train_images_dir)
print(' val_images_dir:  ', val_images_dir)
print(' test_images_dir: ', test_images_dir)

assert train_images_dir is not None, 'Could not find train/images'
assert val_images_dir is not None, 'Could not find valid/images or val/images'
assert test_images_dir is not None, 'Could not find test/images'

def images_in_dir(images_dir: Path):
    exts = ['*.jpg','*.jpeg','*.png','*.bmp','*.webp']
    files = []
    for e in exts:
        files.extend(glob.glob(str(images_dir / e)))
    return sorted(files)

def labels_dir_from_images_dir(images_dir: Path):
    return images_dir.parent / 'labels'

def label_path_for_image(image_path: Path, labels_dir: Path):
    return labels_dir / (image_path.stem + '.txt')

def parse_yolo_label_file(label_file: Path):
    boxes = []
    if not label_file.exists():
        return boxes
    txt = label_file.read_text().strip()
    if not txt:
        return boxes
    for line in txt.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:5])
        boxes.append((cls, x, y, w, h))
    return boxes

def build_annotations_df(split_name: str, images_dir: Path):
    img_files = [Path(p) for p in images_in_dir(images_dir)]
    labels_dir = labels_dir_from_images_dir(images_dir)

    rows = []
    img_rows = []

    for img_path in img_files:
        lp = label_path_for_image(img_path, labels_dir)
        boxes = parse_yolo_label_file(lp)

        img_rows.append({
            'split': split_name,
            'image_path': str(img_path),
            'label_path': str(lp),
            'n_objects': len(boxes),
        })

        for (cls, x, y, w, h) in boxes:
            rows.append({
                'split': split_name,
                'image_path': str(img_path),
                'class_id': cls,
                'class_name': names[cls] if cls < len(names) else str(cls),
                'x_center': x,
                'y_center': y,
                'w': w,
                'h': h,
                'area': w*h,
                'aspect_ratio': (w/h) if h > 0 else None
            })

    return pd.DataFrame(rows), pd.DataFrame(img_rows)

ann_train, imgs_train = build_annotations_df('train', train_images_dir)
ann_val, imgs_val = build_annotations_df('val', val_images_dir)
ann_test, imgs_test = build_annotations_df('test', test_images_dir)

ann = pd.concat([ann_train, ann_val, ann_test], ignore_index=True)
imgs = pd.concat([imgs_train, imgs_val, imgs_test], ignore_index=True)

print('Images:', len(imgs))
print('Annotations (boxes):', len(ann))
ann.head()


In [ ]:
# ============================================
# 5) Add image metadata (width, height, brightness, contrast)
#    (This loops through images once; OK for ~1.5k images on Kaggle CPU)
# ============================================
def image_stats(image_path: str):
    im = cv2.imread(image_path)
    if im is None:
        return None
    h, w = im.shape[:2]
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(gray.mean())
    contrast = float(gray.std())
    # channel means in RGB order for human sanity
    b, g, r = cv2.split(im)
    return {
        "img_w": int(w),
        "img_h": int(h),
        "img_aspect": float(w / h) if h else np.nan,
        "brightness": mean_brightness,
        "contrast": contrast,
        "mean_r": float(r.mean()),
        "mean_g": float(g.mean()),
        "mean_b": float(b.mean()),
    }

img_meta = []
for p in imgs["image_path"].tolist():
    st = image_stats(p)
    if st is None:
        st = {"img_w": np.nan, "img_h": np.nan, "img_aspect": np.nan,
              "brightness": np.nan, "contrast": np.nan,
              "mean_r": np.nan, "mean_g": np.nan, "mean_b": np.nan}
    st["image_path"] = p
    img_meta.append(st)

img_meta = pd.DataFrame(img_meta)
imgs = imgs.merge(img_meta, on="image_path", how="left")

# Attach width/height to each annotation row too
ann = ann.merge(imgs[["image_path","img_w","img_h"]], on="image_path", how="left")

# Pixel coordinates for boxes (useful for some viz)
ann["x_center_px"] = ann["x_center"] * ann["img_w"]
ann["y_center_px"] = ann["y_center"] * ann["img_h"]
ann["w_px"] = ann["w"] * ann["img_w"]
ann["h_px"] = ann["h"] * ann["img_h"]
ann["area_px"] = ann["w_px"] * ann["h_px"]

print("Done.")
imgs.head()

In [ ]:
# ============================================
# 6) Data quality checks
# ============================================
invalid_class_ids = ann[~ann['class_id'].between(0, len(names)-1)]
out_of_bounds = ann[(ann[['x_center','y_center','w','h']] < 0).any(axis=1) | (ann[['x_center','y_center','w','h']] > 1).any(axis=1)]
duplicate_images = imgs['image_path'].duplicated().sum()

quality_report = pd.DataFrame({
    'check': [
        'total_images',
        'total_annotations',
        'duplicate_image_rows',
        'invalid_class_id_rows',
        'out_of_bounds_yolo_rows'
    ],
    'value': [
        len(imgs),
        len(ann),
        int(duplicate_images),
        int(len(invalid_class_ids)),
        int(len(out_of_bounds))
    ]
})

quality_report


## Visual Diagnostics

The following plots validate split balance, class distribution, object geometry, and imaging conditions.


In [ ]:
def show_title(title):
    plt.title(title, pad=12, fontweight="bold")

def fig_ok(figsize=(10,6)):
    plt.figure(figsize=figsize)
    plt.grid(True, alpha=0.2)

In [ ]:
fig_ok()
sns.countplot(data=imgs, x="split")
show_title("1) Images per split")
plt.show()

In [ ]:
fig_ok()
sns.countplot(data=ann, x="split")
show_title("2) Total bounding boxes per split")
plt.show()

In [ ]:
fig_ok((12,6))
class_counts = ann["class_name"].value_counts()
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.xticks(rotation=25, ha="right")
show_title("3) Object count per class (overall)")
plt.show()

In [ ]:
fig_ok((14,6))
tmp = ann.groupby(["split","class_name"]).size().reset_index(name="count")
sns.barplot(data=tmp, x="class_name", y="count", hue="split")
plt.xticks(rotation=25, ha="right")
show_title("4) Object count per class by split")
plt.show()

In [ ]:
fig_ok()
imgs["has_objects"] = imgs["n_objects"] > 0
counts = imgs["has_objects"].value_counts().rename(index={True:"Has objects", False:"No objects"})
plt.pie(counts.values, labels=counts.index, autopct="%1.1f%%")
show_title("5) Images with vs without annotations")
plt.show()

In [ ]:
fig_ok()
sns.histplot(imgs["n_objects"], bins=30)
show_title("6) Distribution: number of objects per image")
plt.xlabel("Objects per image")
plt.show()

In [ ]:
fig_ok()
sns.histplot(ann["w"], bins=40)
show_title("7) Bounding box width (normalized)")
plt.xlabel("w (0..1)")
plt.show()

In [ ]:
fig_ok()
sns.histplot(ann["h"], bins=40)
show_title("8) Bounding box height (normalized)")
plt.xlabel("h (0..1)")
plt.show()

In [ ]:
fig_ok()
sns.histplot(ann["area"], bins=50, log_scale=(False, True))
show_title("9) Bounding box area (normalized, log y)")
plt.xlabel("area")
plt.show()

In [ ]:
fig_ok()
sns.histplot(ann["aspect_ratio"].replace([np.inf, -np.inf], np.nan).dropna(), bins=50)
show_title("10) Bounding box aspect ratio (w/h)")
plt.xlabel("aspect ratio")
plt.show()

In [ ]:
fig_ok((14,6))
sns.boxplot(data=ann, x="class_name", y="area", showfliers=False)
plt.xticks(rotation=25, ha="right")
show_title("11) Box area by class (normalized)")
plt.show()

In [ ]:
fig_ok((8,6))
plt.hist2d(ann["x_center"], ann["y_center"], bins=40)
plt.colorbar(label="count")
plt.gca().invert_yaxis()
show_title("12) Heatmap of box centers (normalized)")
plt.xlabel("x_center"); plt.ylabel("y_center")
plt.show()

In [ ]:
fig_ok()
sns.histplot(ann["x_center"], bins=40)
show_title("13) Distribution of x_center")
plt.show()

In [ ]:
fig_ok()
sns.histplot(ann["y_center"], bins=40)
show_title("14) Distribution of y_center")
plt.show()

In [ ]:
fig_ok((12,6))
res_counts = imgs.groupby(["img_w","img_h"]).size().sort_values(ascending=False).head(15)
labels = [f"{w}x{h}" for (w,h) in res_counts.index]
sns.barplot(x=labels, y=res_counts.values)
plt.xticks(rotation=30, ha="right")
show_title("15) Top image resolutions (Top 15)")
plt.ylabel("count")
plt.show()

In [ ]:
fig_ok()
sns.histplot(imgs["img_aspect"].dropna(), bins=40)
show_title("16) Image aspect ratio distribution (w/h)")
plt.show()

In [ ]:
fig_ok()
sns.histplot(imgs["brightness"].dropna(), bins=40)
show_title("17) Image brightness distribution (mean gray)")
plt.show()

In [ ]:
fig_ok()
sns.histplot(imgs["contrast"].dropna(), bins=40)
show_title("18) Image contrast distribution (std gray)")
plt.show()

In [ ]:
fig_ok((12,6))
sns.kdeplot(imgs["mean_r"].dropna(), label="R")
sns.kdeplot(imgs["mean_g"].dropna(), label="G")
sns.kdeplot(imgs["mean_b"].dropna(), label="B")
show_title("19) Mean RGB channel density")
plt.xlabel("channel mean (0..255 approx)")
plt.legend()
plt.show()

In [ ]:
fig_ok((8,6))
sample = ann.sample(min(len(ann), 5000), random_state=RANDOM_SEED)  # keep plot light
plt.scatter(sample["w"], sample["h"], alpha=0.3)
show_title("20) Scatter: box width vs height (normalized)")
plt.xlabel("w"); plt.ylabel("h")
plt.show()

In [ ]:
# Tiny threshold: area < 0.01 (tune as needed)
tiny_thr = 0.01
tmp = ann.assign(is_tiny=ann["area"] < tiny_thr).groupby("class_name")["is_tiny"].mean().sort_values(ascending=False)

fig_ok((12,6))
sns.barplot(x=tmp.index, y=tmp.values)
plt.xticks(rotation=25, ha="right")
show_title(f"21) Tiny-object rate per class (area < {tiny_thr})")
plt.ylabel("fraction tiny")
plt.show()

In [ ]:
fig_ok((10,6))
sns.boxplot(data=imgs, x="split", y="n_objects")
show_title("22) Objects per image by split")
plt.show()

In [ ]:
# Build image-level presence matrix
presence = (ann.groupby(['image_path','class_name']).size()
              .unstack(fill_value=0)
              .astype(bool)
              .astype(int))

if presence.shape[1] == 0:
    print('No annotation classes available for co-occurrence heatmap.')
else:
    co = presence.T.dot(presence)
    fig_ok((10,8))
    sns.heatmap(co, annot=True, fmt='d', cmap='viridis')
    show_title('23) Class co-occurrence matrix (counts of images where both appear)')
    plt.show()


In [ ]:
fig_ok((8,6))
plt.hist2d(ann["x_center"], ann["y_center"], bins=60)
plt.colorbar(label="count")
plt.gca().invert_yaxis()
show_title("24) Global bounding box density map")
plt.xlabel("x_center"); plt.ylabel("y_center")
plt.show()

In [ ]:
def draw_boxes_on_image(image_path: str, labels_path: str, max_boxes=50):
    im = cv2.imread(image_path)
    if im is None:
        print(f'Skipping unreadable image: {image_path}')
        return

    im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]

    boxes = parse_yolo_label_file(Path(labels_path))[:max_boxes]

    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(im)
    ax.axis('off')

    for (cls, xc, yc, bw, bh) in boxes:
        x1 = (xc - bw/2) * w
        y1 = (yc - bh/2) * h
        rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        label = names[cls] if cls < len(names) else str(cls)
        ax.text(x1, y1-3, label, color='lime', fontsize=12, weight='bold')

    ax.set_title(f'25) Sample: {Path(image_path).name} | boxes={len(boxes)}', fontweight='bold')
    plt.show()

if len(imgs) == 0:
    print('No images available for sample visualization.')
else:
    sample_imgs = imgs.sample(min(len(imgs), 6), random_state=RANDOM_SEED)
    for _, row in sample_imgs.iterrows():
        draw_boxes_on_image(row['image_path'], row['label_path'])


## Automated Findings

Use these generated summaries to quickly extract actionable training decisions.


In [ ]:
# ============================================
# FINAL CELL — Automatic EDA Findings Summary
# ============================================

def eda_summary(ann, imgs, tiny_threshold=0.01):
    if len(imgs) == 0:
        print('No images available. Check dataset paths and rerun.')
        return

    print('='*80)
    print('HOSPITAL OBJECT DETECTION DATASET - EDA SUMMARY REPORT')
    print('='*80)

    total_images = len(imgs)
    total_boxes = len(ann)
    splits = imgs['split'].value_counts()

    print(f'\nTotal Images: {total_images}')
    print(f'Total Annotations (Bounding Boxes): {total_boxes}')
    print('\nSplit Distribution:')
    print(splits.to_string())

    if len(ann) == 0:
        print('\nNo annotations found. Remaining annotation-specific analysis skipped.')
        return

    class_counts = ann['class_name'].value_counts()
    imbalance_ratio = class_counts.max() / class_counts.min() if len(class_counts) > 1 else 1
    mean_area = ann['area'].mean()
    tiny_fraction = (ann['area'] < tiny_threshold).mean()
    avg_objects = imgs['n_objects'].mean()
    max_objects = imgs['n_objects'].max()
    x_mean = ann['x_center'].mean()
    y_mean = ann['y_center'].mean()
    brightness_mean = imgs['brightness'].mean()
    contrast_mean = imgs['contrast'].mean()
    brightness_std = imgs['brightness'].std()

    print('\nClass counts:')
    print(class_counts.to_string())
    print(f'\nImbalance Ratio (max/min): {imbalance_ratio:.2f}')
    print(f'Average normalized box area: {mean_area:.4f}')
    print(f'Tiny objects (area < {tiny_threshold}): {tiny_fraction*100:.2f}%')
    print(f'Average objects per image: {avg_objects:.2f}')
    print(f'Max objects in one image: {max_objects}')
    print(f'Mean center: ({x_mean:.3f}, {y_mean:.3f})')
    print(f'Average brightness: {brightness_mean:.2f}')
    print(f'Average contrast: {contrast_mean:.2f}')

    print('\nRecommendations:')
    print('- Model: YOLOv8m')
    print('- Resolution:', 'imgsz=960' if tiny_fraction > 0.3 else 'imgsz=640')
    if imbalance_ratio > 5:
        print('- Monitor per-class recall and consider balancing strategies.')
    if brightness_std < 10:
        print('- Enable brightness/HSV augmentation.')

    print('\nEDA complete.')
    print('='*80)

eda_summary(ann, imgs)


In [ ]:
# ============================================
# FINAL CELL — Generate Markdown EDA Summary
# ============================================

def generate_markdown_summary(ann, imgs, tiny_threshold=0.01, save_path=None):
    if len(imgs) == 0:
        markdown = '# EDA Summary\n\nNo images available. Check dataset path configuration.'
        print(markdown)
        return markdown

    total_images = len(imgs)
    total_boxes = len(ann)
    splits = imgs['split'].value_counts()

    if len(ann) == 0:
        markdown = f'''# Hospital Object Detection Dataset - EDA Summary\n\n- Total Images: {total_images}\n- Total Annotations: {total_boxes}\n\nNo annotations found, so class/object-size analysis is unavailable.'''
        print(markdown)
        if save_path:
            with open(save_path, 'w') as f:
                f.write(markdown)
        return markdown

    class_counts = ann['class_name'].value_counts()
    imbalance_ratio = class_counts.max() / class_counts.min() if len(class_counts) > 1 else 1
    mean_area = ann['area'].mean()
    tiny_fraction = (ann['area'] < tiny_threshold).mean()
    avg_objects = imgs['n_objects'].mean()
    max_objects = imgs['n_objects'].max()
    x_mean = ann['x_center'].mean()
    y_mean = ann['y_center'].mean()
    brightness_mean = imgs['brightness'].mean()
    brightness_std = imgs['brightness'].std()
    contrast_mean = imgs['contrast'].mean()

    markdown = f'''# Hospital Object Detection Dataset - EDA Summary\n\n## Overview\n- Total Images: {total_images}\n- Total Annotations: {total_boxes}\n\n### Split Distribution\n{splits.to_string()}\n\n## Class Distribution\n{class_counts.to_string()}\n- Imbalance Ratio (max/min): {imbalance_ratio:.2f}\n\n## Object Size\n- Mean normalized area: {mean_area:.4f}\n- Tiny objects (< {tiny_threshold}): {tiny_fraction*100:.2f}%\n\n## Scene Complexity\n- Mean objects/image: {avg_objects:.2f}\n- Max objects/image: {max_objects}\n\n## Spatial and Lighting\n- Mean center: ({x_mean:.3f}, {y_mean:.3f})\n- Brightness mean/std: {brightness_mean:.2f}/{brightness_std:.2f}\n- Contrast mean: {contrast_mean:.2f}\n'''

    print(markdown)
    if save_path:
        with open(save_path, 'w') as f:
            f.write(markdown)
        print(f'\nMarkdown summary saved to: {save_path}')
    return markdown

generate_markdown_summary(ann, imgs)
